# GPS OD-Flow Training (Our Model)

Train **our GPS-based model** (GPS-GNN encoder + TransFlower/Bilinear decoder)
with the selected high-probability experiment grid. Weights and metrics saved to `results/`
and consumed by `benchmark.ipynb` for comparison against baselines.

**Architecture:** GPS-GNN encoder (GINEConv + multi-head attention) with three decoder options:
- `transflower`: TransFlower decoder (Transformer-based)
- `bilinear`: Bilinear decoder
- `gravity_guided`: Gravity-guided decoder

**Experiment axes:** decoder, valid loss/normalization/log combinations, `pe_type` in {`lape`, `None`}, `gps_norm_type`, and the dedicated BL+CE+RLE run.

The GPS-GAN section adds `training_mode='gan'`, an OD random-walk discriminator, and WGAN-GP adversarial training.


In [1]:
import sys, os, gc, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

sys.path.insert(0, str(Path('.').resolve()))
warnings.filterwarnings('ignore')

from models.GPS.config import (
    TrainingConfig, device, cleanup_gpu, MC_EPOCHS,
    SINGLE_CITY_ID, SINGLE_CITY_IDS, MULTI_CITY_IDS, RESULTS_DIR,
    METRICS_CSV, METRICS_VAL_LOSS_CSV, METRICS_CPC_NZ_BEST_CSV, WEIGHTS_DIR,
    ensure_dirs,
)
from models.GPS.data_load import prepare_single_city_data, prepare_multi_city_data
from models.GPS.main import train_single_city, train_multi_city
from models.GPS.metrics import evaluate_full_matrix, compute_metrics
from benchmarking.repeats import single_city_run_id

ensure_dirs()
print(f"Device: {device}")
print(f"Results dir: {RESULTS_DIR}")


Device: cuda:1
Results dir: /home/rk/Документы/Python Projects/GSP_OD_Prediction/results


## GPS Configurations

Edit this cell to select which experiments to run.

In [2]:
from dataclasses import replace

# Config selectors -----------------------------------------------------------
# Empty RUN_ONLY_CONFIG_IDS means "use all enabled configs".
ENABLE_SC_CONFIGS = False
ENABLE_MC_CONFIGS = False
RUN_ONLY_CONFIG_IDS = set()
DISABLED_CONFIG_IDS = set()

EXPERIMENT_EPOCHS = 300

# Experiment policy ----------------------------------------------------------
# Single-city: BatchNorm everywhere.
# Multi-city: GraphNorm everywhere.
# PE: only LaPE and no PE.
# Destination sampling: off everywhere; every row sees all eligible destinations for the active split.
# Pair split: default is all_pairs; switch DEFAULT_PAIR_SPLIT_MODE to compare nonzero_pairs.
# CE/Focal are distribution losses: normalized, no log.
# Huber/MAE/Multitask are regression losses: use normalized + log_norm as the most likely stable variant.
# ZINB is count-style: raw, no log.
DEFAULT_PAIR_SPLIT_MODE = 'all_pairs'
BASE_SPLIT_KW = dict(pair_split_mode=DEFAULT_PAIR_SPLIT_MODE)
BASE_TRAIN_KW = dict(epochs=EXPERIMENT_EPOCHS, mc_epochs=EXPERIMENT_EPOCHS)
PE_OPTIONS = (
    ('LAPE', 'lape'),
    ('NOPE', None),
)
DECODER_KW = {
    'BL': dict(decoder_type='bilinear', use_dest_sampling=False),
    'TF': dict(decoder_type='transflower', use_dest_sampling=False),
    'GG': dict(decoder_type='gravity_guided', use_dest_sampling=False),
}
DECODER_CODES = ('BL', 'TF', 'GG')
LOGGABLE_LOSSES = {'huber', 'mae', 'multitask'}
DISTRIBUTION_LOSSES = {'ce', 'focal'}
IMPLEMENTED_LOSSES = {'ce', 'focal', 'huber', 'mae', 'multitask', 'zinb'}

LOSS_CANDIDATES = [
    ('CE_NORM', 'CE', dict(loss_type='ce', prediction_mode='normalized')),
    ('FOCAL_NORM', 'Focal', dict(loss_type='focal', prediction_mode='normalized', focal_gamma=2.0)),
    ('HUBER_LOG_NORM', 'Huber', dict(loss_type='huber', prediction_mode='normalized', use_log_transform=True)),
    ('MAE_LOG_NORM', 'MAE', dict(loss_type='mae', prediction_mode='normalized', use_log_transform=True)),
    ('MULTITASK_LOG_NORM', 'Multitask', dict(loss_type='multitask', prediction_mode='normalized', use_log_transform=True)),
    ('ZINB_RAW', 'ZINBRaw', dict(loss_type='zinb', prediction_mode='raw')),
]


def _validate_supported_combo(cfg):
    """Fail fast for configs that TrainingConfig accepts but GPS code cannot train."""
    if cfg.loss_type not in IMPLEMENTED_LOSSES:
        raise ValueError(f"{cfg.loss_type=} is not implemented in models/GPS/loss.py")

    if cfg.loss_type in DISTRIBUTION_LOSSES:
        if cfg.prediction_mode != 'normalized':
            raise ValueError(f"{cfg.loss_type} must use prediction_mode='normalized'")
        if cfg.use_log_transform:
            raise ValueError(f"{cfg.loss_type} is incompatible with log transform; using normalized CE/Focal without log")

    if cfg.loss_type == 'zinb':
        if cfg.prediction_mode != 'raw':
            raise ValueError("zinb uses its own count decoder path; keep prediction_mode='raw'")
        if cfg.use_log_transform:
            raise ValueError("zinb already predicts counts; keep use_log_transform=False")

    if cfg.use_log_transform and cfg.loss_type not in LOGGABLE_LOSSES:
        raise ValueError(f"log transform is supported only for {sorted(LOGGABLE_LOSSES)}")

    if cfg.prediction_mode == 'normalized' and cfg.loss_type not in DISTRIBUTION_LOSSES | LOGGABLE_LOSSES:
        raise ValueError(f"normalized mode is not configured for {cfg.loss_type}")

    if cfg.use_dest_sampling:
        raise ValueError("This experiment batch keeps use_dest_sampling=False everywhere")

    if cfg.pair_split_mode not in {'all_pairs', 'nonzero_pairs'}:
        raise ValueError(f"Unsupported pair_split_mode={cfg.pair_split_mode!r}")

    return cfg


def _gps_cfg(decoder_code, norm_type, **kw):
    defaults = dict(
        encoder_type='gps',
        pe_type='lape',
        gps_norm_type=norm_type,
        loss_type='ce',
        prediction_mode='normalized',
        use_log_transform=False,
        use_rle=False,
        **BASE_SPLIT_KW,
        **BASE_TRAIN_KW,
        **DECODER_KW[decoder_code],
    )
    defaults.update(kw)
    defaults['use_dest_sampling'] = False
    defaults.setdefault('pair_split_mode', DEFAULT_PAIR_SPLIT_MODE)
    return _validate_supported_combo(TrainingConfig(**defaults))


def _run_name(scope, decoder_code, loss_name, pe_code, cfg):
    pe_name = 'none' if cfg.pe_type is None else cfg.pe_type
    log_tag = '+log_norm' if cfg.use_log_transform and cfg.prediction_mode == 'normalized' else ('+log' if cfg.use_log_transform else '')
    rle_tag = '+RLE' if cfg.use_rle else ''
    base_tag = ' BASE' if decoder_code == 'BL' and pe_code == 'NOPE' and cfg.loss_type == 'ce' and not cfg.use_rle else ''
    return (
        f'{scope}{base_tag} GPS+{decoder_code}+{loss_name}{log_tag}{rle_tag} | '
        f'pe={pe_name} | norm={cfg.gps_norm_type} | zeros=on | sample_off | ep={EXPERIMENT_EPOCHS}'
    )


def _build_gps_configs(scope, norm_type):
    configs = {}
    for decoder_code in DECODER_CODES:
        for pe_code, pe_type in PE_OPTIONS:
            for loss_code, loss_name, overrides in LOSS_CANDIDATES:
                cfg = _gps_cfg(decoder_code, norm_type, pe_type=pe_type, **overrides)
                rid = f'{scope}_{decoder_code}_{loss_code}_{pe_code}'
                configs[rid] = (_run_name(scope, decoder_code, loss_name, pe_code, cfg), cfg)

    # Dedicated RLE experiment requested: GPS+BL+CE+RLE.
    cfg = _gps_cfg(
        'BL',
        norm_type,
        pe_type='lape',
        loss_type='ce',
        prediction_mode='normalized',
        use_rle=True,
    )
    configs[f'{scope}_BL_CE_RLE_LAPE'] = (
        _run_name(scope, 'BL', 'CE', 'LAPE', cfg),
        cfg,
    )
    return configs


def _apply_config_filters(configs):
    if DISABLED_CONFIG_IDS:
        configs = {rid: item for rid, item in configs.items() if rid not in DISABLED_CONFIG_IDS}
    if RUN_ONLY_CONFIG_IDS:
        configs = {rid: item for rid, item in configs.items() if rid in RUN_ONLY_CONFIG_IDS}
    return configs


SC_CONFIGS = _apply_config_filters(
    _build_gps_configs('SC', 'batch_norm') if ENABLE_SC_CONFIGS else {}
)
MC_CONFIGS = _apply_config_filters(
    _build_gps_configs('MC', 'graph_norm') if ENABLE_MC_CONFIGS else {}
)

print(f"Single-city configs: {len(SC_CONFIGS)}")
print(f"Multi-city configs:  {len(MC_CONFIGS)}")
print(f"TF configs: {sum(1 for k in SC_CONFIGS if '_TF_' in k)}")
print(f"BL configs: {sum(1 for k in SC_CONFIGS if '_BL_' in k)}")
print(f"GG configs: {sum(1 for k in SC_CONFIGS if '_GG_' in k)}")
if RUN_ONLY_CONFIG_IDS:
    print(f"RUN_ONLY_CONFIG_IDS active: {sorted(RUN_ONLY_CONFIG_IDS)}")
if DISABLED_CONFIG_IDS:
    print(f"DISABLED_CONFIG_IDS active: {sorted(DISABLED_CONFIG_IDS)}")
print("\nSingle-city:")
for rid, (run_name, cfg) in SC_CONFIGS.items():
    print(f"  {rid:<38} {run_name:<112} {cfg.describe()}")
print("\nMulti-city:")
for rid, (run_name, cfg) in MC_CONFIGS.items():
    print(f"  {rid:<38} {run_name:<112} {cfg.describe()}")


Single-city configs: 0
Multi-city configs:  0
TF configs: 0
BL configs: 0
GG configs: 0

Single-city:

Multi-city:


## Single-City Training

Three explicit cells below train the same single-city configs on three separate cities.
Each run is saved with a `__city_<city_id>` suffix, so weights and metrics do not overwrite each other.


In [3]:
# Single-city data loading with city+pe+split caching
SC_CITY_1, SC_CITY_2, SC_CITY_3 = SINGLE_CITY_IDS
print(f"Single-city benchmark cities: {SINGLE_CITY_IDS}")

_sc_cache = {}

def get_sc_data(area_id, pe_type='rwpe', pair_split_mode=DEFAULT_PAIR_SPLIT_MODE):
    key = (area_id, pe_type, pair_split_mode)
    if key not in _sc_cache:
        print(f"  Loading single-city data (city={area_id}, pe_type={pe_type}, split={pair_split_mode})...")
        _sc_cache[key] = prepare_single_city_data(area_id=area_id, pe_type=pe_type, pair_split_mode=pair_split_mode)
        n_nodes = _sc_cache[key]['num_nodes']
        train_fit_pairs = _sc_cache[key]['train_fit_mask'].sum()
        train_nz_pairs = _sc_cache[key]['train_mask'].sum()
        print(f"  N={n_nodes}, train_fit_pairs={train_fit_pairs}, train_nz_pairs={train_nz_pairs}")
    return _sc_cache[key]


def train_sc_city(area_id):
    city_results = {}
    print(f"\n{'=' * 70}\n  SINGLE-CITY TRAINING FOR {area_id}\n{'=' * 70}")
    for base_run_id, (run_name, cfg) in SC_CONFIGS.items():
        run_id = single_city_run_id(base_run_id, area_id)
        city_data = get_sc_data(area_id, cfg.pe_type, cfg.pair_split_mode)
        try:
            result = train_single_city(
                run_id,
                f"{run_name} [{area_id}]",
                cfg,
                city_data=city_data,
            )
            city_results[run_id] = result
        except Exception as exc:
            print(f"  ERROR {run_id}: {exc}")
        finally:
            cleanup_gpu()
    print(f"\nCompleted: {len(city_results)} single-city runs for {area_id}")
    return city_results


Single-city benchmark cities: ['48201', '17031', '04013']


In [4]:
sc_results_city_1 = train_sc_city(SC_CITY_1)


  SINGLE-CITY TRAINING FOR 48201

Completed: 0 single-city runs for 48201


In [5]:
sc_results_city_2 = train_sc_city(SC_CITY_2)


  SINGLE-CITY TRAINING FOR 17031

Completed: 0 single-city runs for 17031


In [6]:
sc_results_city_3 = train_sc_city(SC_CITY_3)

sc_results = {}
sc_results.update(sc_results_city_1)
sc_results.update(sc_results_city_2)
sc_results.update(sc_results_city_3)

print(f"\nCombined single-city runs: {len(sc_results)}")
print(sorted(sc_results.keys()))



  SINGLE-CITY TRAINING FOR 04013

Completed: 0 single-city runs for 04013

Combined single-city runs: 0
[]


## Multi-City Training

In [7]:
# Multi-city data loading with pe_type+split caching
_mc_cache = {}

def get_mc_data(pe_type='rwpe', pair_split_mode=DEFAULT_PAIR_SPLIT_MODE):
    key = (pe_type, pair_split_mode)
    if key not in _mc_cache:
        print(f"  Loading multi-city data (pe_type={pe_type}, split={pair_split_mode})...")
        city_data_dict, train_ids, val_ids, test_ids = prepare_multi_city_data(pe_type=pe_type, pair_split_mode=pair_split_mode)
        _mc_cache[key] = (city_data_dict, train_ids, val_ids, test_ids)
        print(f"  Train: {train_ids}  Val: {val_ids}  Test: {test_ids}")
    return _mc_cache[key]


In [8]:
mc_results = {}

for run_id, (run_name, cfg) in MC_CONFIGS.items():
    city_data_dict, train_ids, val_ids, test_ids = get_mc_data(cfg.pe_type, cfg.pair_split_mode)
    try:
        result = train_multi_city(
            run_id, run_name, cfg,
            city_data_dict=city_data_dict,
            train_city_ids=train_ids,
            val_city_ids=val_ids,
            test_city_ids=test_ids,
        )
        mc_results[run_id] = result
    except Exception as e:
        print(f"  ERROR {run_id}: {e}")
    finally:
        cleanup_gpu()

print(f"\nCompleted: {len(mc_results)} multi-city runs")



Completed: 0 multi-city runs


## GPS-GAN Training

These runs add an OD random-walk TCN discriminator. The first config is an adapted paper-like GAT-GAN baseline for the current data/task: GAT generator, linear decoder, no positional encoding, no extra normalization, pure WGAN with clipping, 3 graph layers, 64 hidden channels, 8 attention heads, input-matched generator noise, stochastic eval averaging, distance-aware graph/pair features, and `n_critic=5` for the first 300 epochs.


In [ ]:
# GPS-GAN configs -----------------------------------------------------------
ENABLE_GPS_GAN_SC = False
ENABLE_GPS_GAN_MC = True  # optional; GAN multi-city is much heavier
GPS_GAN_CITY_IDS = SINGLE_CITY_IDS
GPS_GAN_RUN_ONLY = set()  # empty set means run all GPS-GAN configs
GPS_GAN_MC_RUN_ONLY = set()

# Non-paper GAN variants below still use the existing CE+Norm path.
# The paper-like GAT-GAN config overrides this to pure adversarial raw generation.
def _gps_gan_cfg(decoder_code='GG', **kw):
    defaults = dict(
        encoder_type='gps',
        decoder_type=DECODER_KW[decoder_code]['decoder_type'],
        training_mode='gan',
        loss_type='ce',
        prediction_mode='normalized',
        use_log_transform=False,
        pe_type='lape',
        gps_norm_type='graph_norm',
        use_dest_sampling=False,
        pair_split_mode=DEFAULT_PAIR_SPLIT_MODE,
        learning_rate=1.5e-4,
        discriminator_lr=1.5e-4,
        weight_decay=3e-4,
        epochs=EXPERIMENT_EPOCHS,
        mc_epochs=EXPERIMENT_EPOCHS,
        patience=30,
        gan_pretrain_epochs=20,
        adv_weight=0.03,
        gan_n_critic=1,
        gan_walk_len=64,
        gan_walk_batch_size=64,
        gan_disc_layers=4,
        gan_disc_dropout=0.05,
    )
    defaults.update(kw)
    defaults['use_dest_sampling'] = False
    defaults.setdefault('pair_split_mode', DEFAULT_PAIR_SPLIT_MODE)
    return _validate_supported_combo(TrainingConfig(**defaults))

PAPER_GAT_GAN_KW = dict(
    encoder_type='gat',
    decoder_type='linear',
    loss_type='mae',
    prediction_mode='raw',
    use_log_transform=False,
    pe_type=None,
    gps_norm_type='none',
    gnn_layers=3,
    gnn_heads=8,
    gan_only=True,
    gan_pretrain_epochs=0,
    gan_regularizer='clip',
    gan_clip_value=0.01,
    gan_gp_lambda=0.0,
    gan_n_critic=5,
    gan_n_critic_after_epoch=300,
    gan_n_critic_after=1,
    gan_noise_dim=0,
    gan_noise_dim_mode='match_input',
    gan_eval_num_samples=4,
    gat_use_edge_attr=True,
    pair_use_distance=True,
    adv_weight=1.0,
    learning_rate=3e-4,
    discriminator_lr=3e-4,
    weight_decay=0.0,
    patience=EXPERIMENT_EPOCHS,
    gan_use_supervised_monitoring=True,
)


PAPER_ODGN_KW = dict(
    **PAPER_GAT_GAN_KW,
)

PAPER_ODGN_KW['decoder_type']='gravity_guided'  # gravity decoder — key difference from GAT-GAN baseline

PURE_GAN_NO_PRETRAIN_KW = dict(
    gan_only=True,
    gan_pretrain_epochs=0,
    adv_weight=1.0,
    learning_rate=3e-4,
    discriminator_lr=1e-4,
    gan_n_critic=1,
    gan_disc_hidden_dim=32,
    gan_disc_layers=2,
    gan_walk_len=32,
    gan_walk_batch_size=128,
    gan_tau=1.5,
)

GPS_GAN_SC_CONFIGS = {
    'SC_GAT_GAN_ORIG_PAPER_RAW_NONE_NOPE': (
        'SC adapted paper-like GAT-GAN+Linear+raw | pe=none | norm=none | pure WGAN-clip | edge_attr=on | dist=on | val logs',
        _gps_gan_cfg('BL', **PAPER_GAT_GAN_KW),
    ),
    # 'SC_GAT_GAN_ORIG_TUNED_CE_NORM_BN_NOPE': (
    #     'SC tuned pure GAT-GAN+Linear+CE-param+Norm | pe=none | batch_norm | no pretrain',
    #     _gps_gan_cfg(
    #         'BL',
    #         encoder_type='gat',
    #         decoder_type='linear',
    #         pe_type=None,
    #         gps_norm_type='batch_norm',
    #         **PURE_GAN_NO_PRETRAIN_KW,
    #     ),
    # ),
    # 'SC_GAT_GAN_ORIG_TUNED_CE_NORM_BN_LAPE': (
    #     'SC tuned pure GAT-GAN+Linear+CE-param+Norm | pe=lape | batch_norm | no pretrain',
    #     _gps_gan_cfg(
    #         'BL',
    #         encoder_type='gat',
    #         decoder_type='linear',
    #         pe_type='lape',
    #         gps_norm_type='batch_norm',
    #         **PURE_GAN_NO_PRETRAIN_KW,
    #     ),
    # ),
    # 'SC_GG_GAN_ONLY_TUNED_CE_NORM_GN_LAPE': (
    #     'SC tuned pure GPS-GAN+GG+CE-param+Norm | pe=lape | graph_norm | no pretrain',
    #     _gps_gan_cfg('GG', pe_type='lape', gps_norm_type='graph_norm', **PURE_GAN_NO_PRETRAIN_KW),
    # ),
    # 'SC_GG_GAN_ONLY_CE_NORM_GN_LAPE': (
    #     'SC GPS-GAN-ONLY+GG+CE+Norm (pretrain supervised, then GAN only) | pe=lape | graph_norm',
    #     _gps_gan_cfg('GG', gan_only=True),
    # ),
    # 'SC_BL_GAN_CE_NORM_GN_LAPE': (
    #     'SC GPS-GAN+BL+CE+Norm (requested Log+CE -> CE no-log) | pe=lape | graph_norm',
    #     _gps_gan_cfg('BL'),
    # ),
    # 'SC_TF_GAN_CE_NORM_GN_LAPE': (
    #     'SC GPS-GAN+TF+CE+Norm (requested Log+CE -> CE no-log) | pe=lape | graph_norm',
    #     _gps_gan_cfg('TF'),
    # ),
    'SC_GG_GAN_CE_NORM_GN_LAPE': (
        'SC GPS-GAN+GG+CE+Norm (requested Log+CE -> CE no-log) | pe=lape | graph_norm',
        _gps_gan_cfg('GG'),
    ),
    'SC_ODGN_PAPER': (
        'SC adapted paper-like ODGN: GAT+gravity-guided+WGAN-clip | no PE | no norm | edge_attr=on | dist=on | val logs',
        _gps_gan_cfg('GG', **PAPER_ODGN_KW),
    ),
}
if GPS_GAN_RUN_ONLY:
    GPS_GAN_SC_CONFIGS = {rid: item for rid, item in GPS_GAN_SC_CONFIGS.items() if rid in GPS_GAN_RUN_ONLY}

def _gps_gan_mc_id(rid):
    return rid.replace('SC_', 'MC_', 1).replace('_BN_', '_GN_')


def _gps_gan_mc_name(name):
    return name.replace('SC ', 'MC ', 1).replace('batch_norm', 'graph_norm')


GPS_GAN_MC_CONFIGS = {
    _gps_gan_mc_id(rid): (
        _gps_gan_mc_name(name),
        replace(
            cfg,
            mc_epochs=EXPERIMENT_EPOCHS,
            gps_norm_type=('graph_norm' if cfg.gps_norm_type != 'none' else 'none'),
        ),
    )
    for rid, (name, cfg) in GPS_GAN_SC_CONFIGS.items()
}
if GPS_GAN_MC_RUN_ONLY:
    GPS_GAN_MC_CONFIGS = {rid: item for rid, item in GPS_GAN_MC_CONFIGS.items() if rid in GPS_GAN_MC_RUN_ONLY}

print(f"GPS-GAN single-city configs: {len(GPS_GAN_SC_CONFIGS)}")
for rid, (run_name, cfg) in GPS_GAN_SC_CONFIGS.items():
    print(f"  {rid:<34} {run_name:<92} {cfg.describe()}")
print(f"GPS-GAN multi-city configs:  {len(GPS_GAN_MC_CONFIGS)}")


In [ ]:
gps_gan_results = {}

def train_gps_gan_sc_city(area_id):
    city_results = {}
    print(f"\n{'=' * 70}\n  GPS-GAN TRAINING FOR {area_id}\n{'=' * 70}")
    for base_run_id, (run_name, cfg) in GPS_GAN_SC_CONFIGS.items():
        run_id = single_city_run_id(base_run_id, area_id)
        city_data = get_sc_data(area_id, cfg.pe_type, cfg.pair_split_mode)
        try:
            result = train_single_city(
                run_id,
                f"{run_name} [{area_id}]",
                cfg,
                city_data=city_data,
            )
            city_results[run_id] = result
        except Exception as exc:
            print(f"  ERROR {run_id}: {exc}")
        finally:
            cleanup_gpu()
    print(f"\nCompleted: {len(city_results)} GPS-GAN runs for {area_id}")
    return city_results

if ENABLE_GPS_GAN_SC:
    for area_id in GPS_GAN_CITY_IDS:
        gps_gan_results.update(train_gps_gan_sc_city(area_id))

if ENABLE_GPS_GAN_MC:
    for run_id, (run_name, cfg) in GPS_GAN_MC_CONFIGS.items():
        city_data_dict, train_ids, val_ids, test_ids = get_mc_data(cfg.pe_type, cfg.pair_split_mode)
        try:
            result = train_multi_city(
                run_id,
                run_name,
                cfg,
                city_data_dict=city_data_dict,
                train_city_ids=train_ids,
                val_city_ids=val_ids,
                test_city_ids=test_ids,
            )
            gps_gan_results[run_id] = result
        except Exception as exc:
            print(f"  ERROR {run_id}: {exc}")
        finally:
            cleanup_gpu()

print(f"\nCompleted GPS-GAN runs: {len(gps_gan_results)}")
print(sorted(gps_gan_results.keys()))


## GMEL-GPS Training

Single-city GMEL-GPS variants are also trained separately for the same three cities,
with per-city run IDs so `benchmark.ipynb` can load them city by city.


In [ ]:
from models.GPS.config import TrainingConfig
from models.GMEL_GPS.main import train as gmel_gps_train

# GMEL_GPS in this repo is GPS encoders + bilinear pretraining head + GBRT/LGBM tree decoder.
# It does not expose GPS BL/TF/GG pair decoders; using those labels as decoder_type would be misleading.
# Requested Log+CE+Norm is also incompatible, so these use CE+Norm no-log, GraphNorm, LaPE/NoPE, 300 epochs.
def _gmel_gps(**kw):
    defaults = dict(
        encoder_type='gps',
        decoder_type='gbrt',
        loss_type='ce',
        prediction_mode='normalized',
        use_log_transform=False,
        pe_type='lape',
        gps_norm_type='graph_norm',
        use_dest_sampling=False,
        pair_split_mode=DEFAULT_PAIR_SPLIT_MODE,
        epochs=EXPERIMENT_EPOCHS,
        patience=30,
    )
    defaults.update(kw)
    defaults.setdefault('pair_split_mode', DEFAULT_PAIR_SPLIT_MODE)
    return TrainingConfig(**defaults)

GMEL_GPS_CONFIGS = {
    'GMEL_GPS_GBRT_CE_NORM_GN_LAPE': (
        'GMEL-GPS+GBRT+CE+Norm | pe=lape | graph_norm | ep=300',
        _gmel_gps(decoder_type='gbrt', pe_type='lape'),
    ),
    'GMEL_GPS_LGBM_CE_NORM_GN_LAPE': (
        'GMEL-GPS+LGBM+CE+Norm | pe=lape | graph_norm | ep=300',
        _gmel_gps(decoder_type='lgbm', pe_type='lape'),
    ),
    'GMEL_GPS_GBRT_CE_NORM_GN_NONE_BASE': (
        'GMEL-GPS BASE+GBRT+CE+Norm | pe=none | graph_norm | ep=300',
        _gmel_gps(decoder_type='gbrt', pe_type=None),
    ),
}
print(f"GMEL_GPS configs: {len(GMEL_GPS_CONFIGS)}")
for rid, (name, cfg) in GMEL_GPS_CONFIGS.items():
    print(f"  {rid:<38}  {name:<72}  {cfg.describe()}")


In [ ]:
# GMEL-GPS data loading with city+pe+split caching
_gg_cache = {}

def get_gg_data(area_id, pe_type, pair_split_mode=DEFAULT_PAIR_SPLIT_MODE):
    key = (area_id, pe_type, pair_split_mode)
    if key not in _gg_cache:
        print(f"  Loading GMEL-GPS data (city={area_id}, pe_type={pe_type}, split={pair_split_mode})...")
        _gg_cache[key] = prepare_single_city_data(area_id=area_id, pe_type=pe_type, pair_split_mode=pair_split_mode)
        n_nodes = _gg_cache[key]['num_nodes']
        train_fit_pairs = _gg_cache[key]['train_fit_mask'].sum()
        train_nz_pairs = _gg_cache[key]['train_mask'].sum()
        print(f"  N={n_nodes}, train_fit_pairs={train_fit_pairs}, train_nz_pairs={train_nz_pairs}")
    return _gg_cache[key]


def train_gmel_gps_city(area_id):
    city_results = {}
    print(f"\n{'=' * 70}\n  GMEL-GPS TRAINING FOR {area_id}\n{'=' * 70}")
    for base_run_id, (run_name, cfg) in GMEL_GPS_CONFIGS.items():
        run_id = single_city_run_id(base_run_id, area_id)
        city_data = get_gg_data(area_id, cfg.pe_type, cfg.pair_split_mode)
        try:
            result = gmel_gps_train(run_id, f"{run_name} [{area_id}]", cfg, city_data)
            city_results[run_id] = result
        except Exception as exc:
            print(f"  ERROR {run_id}: {exc}")
        finally:
            cleanup_gpu()
    print(f"\nCompleted: {len(city_results)} GMEL-GPS runs for {area_id}")
    return city_results


In [ ]:
gmel_gps_results_city_1 = train_gmel_gps_city(SC_CITY_1)

In [ ]:
gmel_gps_results_city_2 = train_gmel_gps_city(SC_CITY_2)

In [ ]:
gmel_gps_results_city_3 = train_gmel_gps_city(SC_CITY_3)

gmel_gps_results = {}
gmel_gps_results.update(gmel_gps_results_city_1)
gmel_gps_results.update(gmel_gps_results_city_2)
gmel_gps_results.update(gmel_gps_results_city_3)

print(f"\nCombined GMEL-GPS runs: {len(gmel_gps_results)}")
print(sorted(gmel_gps_results.keys()))


In [ ]:
# GMEL-GPS summary is shown below together with the other result tables and plots.


## Results Summary

In [ ]:
def print_results_table(results, label):
    if not results:
        print(f"  No {label} results.")
        return
    print(f"\n{'='*70}")
    print(f"  {label} Results")
    print(f"{'='*70}")
    print(f"  {'Run ID':<28} {'CPC_full':>9} {'CPC_nz':>9} {'CPC_test':>9} {'MAE':>9}  Status")
    print(f"  {'-'*76}")
    for rid, r in results.items():
        mf = r.get('metrics_full', {})
        mnz = r.get('metrics_nonzero', {})
        mt = r.get('metrics_test_pairs', {})
        status = r.get('status', '?')
        print(f"  {rid:<28} {mf.get('CPC', float('nan')):>9.4f} "
              f"{mnz.get('CPC', float('nan')):>9.4f} "
              f"{mt.get('CPC', float('nan')):>9.4f} "
              f"{mf.get('MAE', float('nan')):>9.4f}  {status}")

print_results_table(sc_results, "Single-City")
print_results_table(mc_results, "Multi-City")
print_results_table(globals().get('gps_gan_results', {}), "GPS-GAN")
print_results_table(gmel_gps_results, "GMEL-GPS")

# Show saved metrics CSVs
for metrics_path in (METRICS_VAL_LOSS_CSV, METRICS_CPC_NZ_BEST_CSV, METRICS_CSV):
    if metrics_path.exists():
        df = pd.read_csv(metrics_path)
        print(f"\n  Total rows in {metrics_path.name}: {len(df)}")
print(f"  Saved weights: {len(list(WEIGHTS_DIR.glob('*.pt')))}")


In [ ]:
# Training curves
def plot_histories(results, title):
    if not results:
        return
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(title)
    for rid, r in results.items():
        h = r.get('history', {})
        if not h:
            continue
        axes[0].plot(h.get('train_loss', []), label=rid, alpha=0.8)
        axes[1].plot(h.get('val_loss', []), label=rid, alpha=0.8)
        axes[2].plot(h.get('val_cpc_full', []), label=rid, alpha=0.8)
    axes[0].set_title('Train Loss'); axes[0].set_xlabel('Epoch')
    axes[1].set_title('Val Loss');   axes[1].set_xlabel('Epoch')
    axes[2].set_title('CPC (full)'); axes[2].set_xlabel('Epoch')
    for ax in axes:
        ax.legend(fontsize=7, ncol=2)
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_histories(sc_results, "Single-City Training Histories")
plot_histories(mc_results, "Multi-City Training Histories")
plot_histories(globals().get('gps_gan_results', {}), "GPS-GAN Training Histories")
plot_histories(gmel_gps_results, "GMEL-GPS Training Histories")
